In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "6"

import torch

from tts.config.ndaligner.training_module_config import NDAlignerTrainingModuleConfigs
from tts.config.utils.io import load_config
from tts.models.ndaligner import init_nd_aligner_training_module

/home/blue2959/monotonic_tts/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## INIT Models

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
aligner_training_module_cfg_path = "./runs/nd_aligner_vctk_10ms_coupling_dec_20260715-122003/model_config.json"
aligner_training_module_ckpt_path = "./runs/nd_aligner_vctk_10ms_coupling_dec_20260715-122003/checkpoints_timit_bae/best_step_timit_bae_0.020846_step_577500_epoch_33.pth"


model_config = load_config(aligner_training_module_cfg_path, NDAlignerTrainingModuleConfigs,)

aligner_training_module = init_nd_aligner_training_module(
    config=model_config,
    device=device,
)
aligner_training_module.load_checkpoint(
    ckpt_path=aligner_training_module_ckpt_path,
    device=device,
)
aligner = aligner_training_module.nd_aligner.eval()

Loading nested state_dict from key 'model' in ./runs/nd_aligner_vctk_10ms_coupling_dec_20260715-122003/checkpoints_timit_bae/best_step_timit_bae_0.020846_step_577500_epoch_33.pth
⚠️ Checkpoint load summary (strict=False):
  - Unexpected top-level modules: ['nd_aligner', 'vocoder']
Checkpoint loading process finished.


## INIT BenchMarkers (TIMIT)

In [3]:
from tts.benchmark.timit.benchmarker import TIMITBenchMarker

TIMIT_ROOT = "/shared/data_zfs/blue2959/TIMIT/TEST"
assert aligner.input_maker is not None

timit_benchmarker = TIMITBenchMarker(
    root_dir=TIMIT_ROOT,
    ref_audio_sr=16_000,
    hyp_audio_sr=22_050,
    hyp_hop_length=256,
    input_maker=aligner.input_maker,
    hyp_ignore_symbols=aligner.input_maker.tokenizer.ignore_symbols,
    max_ref_words_per_hyp_word=5,
)

[TIMITBenchMarker] Found 1680 valid (WRD, WAV, TXT) triplets.


In [4]:
with torch.no_grad():
    metrics = timit_benchmarker(
        aligner=aligner,
        vocoder=None,
        max_test_samples=None,
    )

print(metrics.word_boundary_error)
print(metrics.p_word_100ms)
print(metrics.p_word_50ms)
print(metrics.p_word_25ms)
print(metrics.p_word_10ms)

Computing Alignments: 100%|██████████| 1680/1680 [01:05<00:00, 25.53it/s]

0.02030583657324314
98.65414500236511
91.7461097240448
73.88896942138672
39.219123125076294


## INIT BenchMarkers (Buckeye)

In [5]:
from tts.benchmark.timit.benchmarker import TIMITBenchMarker
assert aligner.input_maker is not None


BUCKEYE_ROOT = "/shared/data_zfs/blue2959/Buckeye-grid" # (compatible with timit benchmarker!)

buckeye_benchmarker = TIMITBenchMarker(
    root_dir=BUCKEYE_ROOT,
    ref_audio_sr=16_000,
    hyp_audio_sr=22_050,
    hyp_hop_length=256,
    input_maker=aligner.input_maker,
    hyp_ignore_symbols=aligner.input_maker.tokenizer.ignore_symbols,
    max_ref_words_per_hyp_word=5,
)

[TIMITBenchMarker] Found 19273 valid (WRD, WAV, TXT) triplets.


In [ ]:
with torch.inference_mode():
    metrics = buckeye_benchmarker(
        aligner=aligner,
        vocoder=None,
        max_test_samples=None,
    )

print(metrics.word_boundary_error)
print(metrics.p_word_100ms)
print(metrics.p_word_50ms)
print(metrics.p_word_25ms)
print(metrics.p_word_10ms)

Computing Alignments: 100%|██████████| 5000/5000 [03:29<00:00, 23.87it/s]

0.024751178920269012
95.26250958442688
89.64169025421143
74.58143830299377
40.08335769176483
